# 05 – Reinforcement Learning: DQN für CartPole

**Lernziele:**
- Grundkonzepte des Reinforcement Learning (Agent, Environment, Reward)
- Deep Q-Network (DQN) mit PyTorch implementieren
- Experience Replay Buffer für stabiles Training
- Target Network für reduzierte Korrelation
- Epsilon-Greedy Exploration
- Double DQN (Policy-Netz wählt, Target-Netz bewertet)

---

In [ ]:
import random
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim

# Reproduzierbarkeit
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Verwende Gerät: {DEVICE}")

## 1. Environment: CartPole-v1

**CartPole** ist das "Hello World" des Reinforcement Learning:
- **Ziel:** Einen Stab auf einem Wagen balancieren
- **State (4):** Wagenposition, Wagen-Geschwindigkeit, Stab-Winkel, Stab-Winkelgeschwindigkeit
- **Actions (2):** Links schieben (0), Rechts schieben (1)
- **Reward:** +1 für jeden Zeitschritt, in dem der Stab nicht fällt
- **Gelöst:** Durchschnittlicher Reward ≥ 195 über 100 Episoden

In [ ]:
try:
    import gymnasium as gym
except ImportError:
    print("Installiere gymnasium...")
    import subprocess
    subprocess.check_call(["uv", "pip", "install", "gymnasium"])
    import gymnasium as gym

env = gym.make("CartPole-v1")
STATE_DIM = env.observation_space.shape[0]   # 4
ACTION_DIM = env.action_space.n              # 2

print("Environment: CartPole-v1")
print(f"State-Dimension: {STATE_DIM}, Action-Dimension: {ACTION_DIM}")

## 2. Hyperparameter

In [ ]:
EPISODES = 500
BATCH_SIZE = 64
GAMMA = 0.99                # Discount-Faktor
EPSILON_START = 1.0         # Start-Exploration
EPSILON_END = 0.01          # Minimale Exploration
EPSILON_DECAY = 0.995       # Decay pro Episode
LEARNING_RATE = 0.001
TARGET_UPDATE = 10          # Target-Network Update-Frequenz (Episoden)
MEMORY_SIZE = 10_000        # Replay-Buffer-Größe
MIN_MEMORY = 1_000          # Mindest-Erfahrungen vor Training

## 3. Q-Network

Ein einfaches Feed-Forward-Netz, das Q-Werte für jede Aktion approximiert:
- Input: State (4 Werte)
- Hidden: 128 → 128 (ReLU)
- Output: Q-Wert pro Aktion (2 Werte)

**Q-Wert:** Erwarteter kumulativer Reward, wenn wir in diesem State Aktion `a` wählen.

In [ ]:
class QNetwork(nn.Module):
    """Einfaches Feed-Forward-Netz für Q-Wert-Approximation."""

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

## 4. Replay Buffer

Speichert Erfahrungen $(s, a, r, s', done)$ und sampelt zufällige Batches.

**Warum?**
- Bricht Korrelation zwischen aufeinanderfolgenden Samples
- Ermöglicht Wiederverwendung von Erfahrungen
- Stabilisiert das Training

In [ ]:
class ReplayBuffer:
    """Experience Replay Buffer mit zufälligem Sampling."""

    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(DEVICE),
            torch.LongTensor(actions).to(DEVICE),
            torch.FloatTensor(rewards).to(DEVICE),
            torch.FloatTensor(np.array(next_states)).to(DEVICE),
            torch.FloatTensor(dones).to(DEVICE),
        )

    def __len__(self) -> int:
        return len(self.buffer)

## 5. DQN-Agent

Der Agent kombiniert:
- **Policy Network:** Wählt Aktionen und wird trainiert
- **Target Network:** Stabiler Klon für TD-Target-Berechnung
- **Epsilon-Greedy:** Balanciert Exploration vs. Exploitation
- **Double DQN:** Policy-Netz wählt die beste Aktion, Target-Netz bewertet sie

**Bellman-Update:** $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$

In [ ]:
class DQNAgent:
    """DQN-Agent mit Target-Network und Epsilon-Greedy."""

    def __init__(self, state_dim: int, action_dim: int):
        self.action_dim = action_dim

        self.policy_net = QNetwork(state_dim, action_dim).to(DEVICE)
        self.target_net = QNetwork(state_dim, action_dim).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.memory = ReplayBuffer(MEMORY_SIZE)
        self.epsilon = EPSILON_START

    def select_action(self, state: np.ndarray, evaluate: bool = False) -> int:
        """Epsilon-Greedy Action Selection."""
        if evaluate or random.random() > self.epsilon:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
                q_values = self.policy_net(state_t)
                return q_values.argmax(dim=1).item()
        else:
            return random.randrange(self.action_dim)

    def update(self) -> float | None:
        """Ein Trainingsschritt mit Experience Replay."""
        if len(self.memory) < MIN_MEMORY:
            return None

        states, actions, rewards, next_states, dones = self.memory.sample(BATCH_SIZE)

        # Current Q values
        q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q values (Double DQN)
        with torch.no_grad():
            next_actions = self.policy_net(next_states).argmax(dim=1)
            next_q_values = self.target_net(next_states).gather(
                1, next_actions.unsqueeze(1)
            ).squeeze(1)
            target_q_values = rewards + GAMMA * next_q_values * (1 - dones)

        loss = nn.MSELoss()(q_values, target_q_values)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        return loss.item()

    def update_target(self):
        """Target-Network auf Policy-Network synchronisieren."""
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def decay_epsilon(self):
        """Epsilon exponentiell reduzieren."""
        self.epsilon = max(EPSILON_END, self.epsilon * EPSILON_DECAY)


agent = DQNAgent(STATE_DIM, ACTION_DIM)
print(f"Policy-Network Parameter: {sum(p.numel() for p in agent.policy_net.parameters()):,}")

## 6. Training

Der Trainingsloop:
1. Environment resetten
2. Aktion wählen (Epsilon-Greedy)
3. Aktion ausführen, Reward & nächsten State erhalten
4. Erfahrung im Replay Buffer speichern
5. Batch sampeln & Netzwerk updaten
6. Epsilon decay, Target-Network periodisch synchronisieren

In [ ]:
episode_rewards: list[float] = []
epsilons: list[float] = []
losses: list[float] = []
moving_avg: list[float] = []

print("── Training ──")
for episode in range(1, EPISODES + 1):
    state, _ = env.reset()
    episode_reward = 0.0
    episode_losses: list[float] = []

    while True:
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.memory.push(state, action, reward, next_state, done)
        state = next_state
        episode_reward += reward

        loss = agent.update()
        if loss is not None:
            episode_losses.append(loss)

        if done:
            break

    episode_rewards.append(episode_reward)
    epsilons.append(agent.epsilon)
    avg_loss = np.mean(episode_losses) if episode_losses else 0.0
    losses.append(avg_loss)

    agent.decay_epsilon()

    if episode % TARGET_UPDATE == 0:
        agent.update_target()

    # Gleitender Durchschnitt (letzte 100 Episoden)
    if len(episode_rewards) >= 100:
        ma = np.mean(episode_rewards[-100:])
    else:
        ma = np.mean(episode_rewards)
    moving_avg.append(ma)

    if episode % 50 == 0 or episode == 1:
        print(f"Episode {episode:4d}/{EPISODES} | Reward: {episode_reward:6.1f} | "
              f"Avg100: {ma:6.1f} | Epsilon: {agent.epsilon:.3f} | Loss: {avg_loss:.4f}")

env.close()

print(f"\nFinaler Avg100-Reward: {moving_avg[-1]:.2f}")
solved = moving_avg[-1] >= 195.0
print(f"CartPole {'GELÖST! 🎉' if solved else 'noch nicht gelöst (Ziel: 195.0)'}")

## 7. Visualisierung

Vier Plots: Reward-Verlauf, Epsilon-Decay, Training Loss, Reward-Verteilung.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Reward pro Episode
axes[0, 0].plot(episode_rewards, alpha=0.4, color="blue", linewidth=0.8,
                label="Episode Reward")
axes[0, 0].plot(moving_avg, color="red", linewidth=2,
                label="Moving Avg (100)")
axes[0, 0].axhline(y=195.0, color="green", linestyle="--",
                   label="Solved (195)")
axes[0, 0].set_title("Reward pro Episode")
axes[0, 0].set_xlabel("Episode")
axes[0, 0].set_ylabel("Reward")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Epsilon-Decay
axes[0, 1].plot(epsilons, color="purple", linewidth=2)
axes[0, 1].set_title("Epsilon-Decay")
axes[0, 1].set_xlabel("Episode")
axes[0, 1].set_ylabel("Epsilon")
axes[0, 1].grid(True, alpha=0.3)

# Loss
axes[1, 0].plot(losses, color="orange", alpha=0.7, linewidth=1)
axes[1, 0].set_title("Training Loss")
axes[1, 0].set_xlabel("Episode")
axes[1, 0].set_ylabel("Loss")
axes[1, 0].grid(True, alpha=0.3)

# Reward-Histogramm
axes[1, 1].hist(episode_rewards, bins=30, color="steelblue", edgecolor="white",
                alpha=0.8)
axes[1, 1].axvline(x=195.0, color="green", linestyle="--", linewidth=2,
                   label="Solved (195)")
axes[1, 1].set_title("Reward-Verteilung")
axes[1, 1].set_xlabel("Reward")
axes[1, 1].set_ylabel("Häufigkeit")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(Path("output") / "05_rl_cartpole_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Evaluation: Beispiel-Trajektorie

Wir lassen den trainierten Agenten eine Episode spielen und zeigen 4 Frames.

In [ ]:
print("── Evaluation: Beispiel-Trajektorie ──")
eval_env = gym.make("CartPole-v1", render_mode="rgb_array")
state, _ = eval_env.reset()
total_reward = 0.0
step_count = 0
frames: list = []

for step in range(500):
    action = agent.select_action(state, evaluate=True)
    state, reward, terminated, truncated, _ = eval_env.step(action)
    total_reward += reward
    step_count += 1
    frames.append(eval_env.render())
    if terminated or truncated:
        break

eval_env.close()
print(f"Evaluations-Reward: {total_reward:.1f} in {step_count} Schritten")

# Zeige 4 Frames der Trajektorie
fig3, axes3 = plt.subplots(1, 4, figsize=(16, 4))
indices = np.linspace(0, len(frames) - 1, 4, dtype=int)
for i, idx in enumerate(indices):
    axes3[i].imshow(frames[idx])
    axes3[i].set_title(f"Step {idx}")
    axes3[i].axis("off")
fig3.suptitle(f"Beispiel-Trajektorie (Reward: {total_reward:.1f})", fontsize=13)
fig3.tight_layout()

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
fig3.savefig(output_dir / "05_rl_cartpole_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Plots gespeichert unter: {output_dir}/")

---
## Zusammenfassung

| Konzept | Beschreibung |
|---|---|
| **Markov Decision Process** | (State, Action, Reward, Next State) – Grundmodell des RL |
| **Q-Learning** | Lerne Q(s,a) – erwarteter Reward für Aktion a in State s |
| **DQN** | Deep Q-Network – approximiert Q-Werte mit neuronalem Netz |
| **Experience Replay** | Speichert & sampelt Erfahrungen für stabiles Training |
| **Target Network** | Eingefrorener Klon des Policy-Netzes für stabile TD-Targets |
| **Double DQN** | Policy-Netz wählt Aktion, Target-Netz bewertet – reduziert Overestimation |
| **Epsilon-Greedy** | Balanciert Exploration (zufällig) vs. Exploitation (greedy) |
| **Gradient Clipping** | Begrenzt Gradienten für stabileres Training |

---

## 🎉 Glückwunsch! Du hast alle 5 Notebooks durchgearbeitet!

| # | Notebook | Thema |
|---|---|---|
| 01 | `01_tensor_basics.ipynb` | Tensoren, Operationen, GPU, Autograd |
| 02 | `02_neural_network.ipynb` | NN mit PyTorch, MLP vs. CNN, Training-Loop |
| 03 | `03_cnn_mnist.ipynb` | CNN mit BatchNorm, Visualisierung |
| 04 | `04_transfer_learning.ipynb` | ResNet18 Fine-Tuning, Data Augmentation |
| 05 | `05_rl_cartpole.ipynb` | DQN, Experience Replay, Target Network |

**Nächste Schritte:**
- Experimentiere mit Hyperparametern
- Probiere eigene Architekturen
- Trainiere auf anderen Datasets (Fashion-MNIST, CIFAR-100)
- Erkunde weitere RL-Algorithmen (PPO, A2C)